# Reading From Bronze

In the silver layer, we will only clean the data for each table. (Merging / data modeling will be done in the Gold Layer)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Transformation


| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| iata_code | The three-letter IATA code for the airport (if it has one). |
| time | The date of the daily aggregate in EST timezone. |
| temperature_2m_max | Highest air temperature of the day, measured 2 m above ground. |
| temperature_2m_min | Lowest air temperature of the day, measured 2 m above ground. |
| rain_sum | Total rain for the day from large-scale weather systems. |
| showers_sum | Total shower precipitation for the day (convective, short and intense bursts). |
| snowfall_sum | Total snowfall for the day. |
| precipitation_sum | Total of all precipitation for the day: rain, showers, and snow (as water equivalent). |
| wind_speed_10m_max | 	Highest sustained wind speed of the day, measured 10 m above ground. |
| wind_gusts_10m_max | Highest wind gust of the day, measured 10 m above ground. |

In [0]:
df_weather = spark.table("bronze_travel.weather")

## Exploring raw data

In [0]:
df_weather.display()

In [0]:
# Check data types

df_weather.dtypes

**Clean Up To Do List:**

- Trim white spaces on every column
- Remove duplicates
- key columns cannot be null: "iata_code", "date"

## Trim white spaces from all columns

In [0]:
for column_name, data_type in df_weather.dtypes:
  if data_type == "string":
    df_weather = df_weather.withColumn(column_name, F.trim(F.col(column_name)))

## Drop Duplicates

In [0]:
# Find groups with more than one row
dup_groups = (
    df_weather.groupBy(*df_weather.columns)
    .count()
    .filter(F.col("count") > 1)
)

# Number of *extra* rows that dropDuplicates will remove
# (count - 1 per group, summed across all groups)
dup_row_count = (
    dup_groups
    .withColumn("extra_rows", F.col("count") - 1)
    .agg(F.sum("extra_rows"))
    .first()[0]
) or 0  # guard against None if there are no duplicates at all

print(f"Found {dup_row_count} duplicate rows across {dup_groups.count()} groups")

before_count = df_weather.count()

# Drop duplicates, keeping one row per group
df_weather = df_weather.dropDuplicates(df_weather.columns)

after_count = df_weather.count()
actual_dropped = before_count - after_count

print(f"Before: {before_count}, After: {after_count}, Dropped: {actual_dropped}")

if actual_dropped != dup_row_count:
    raise ValueError(
        f"Expected to drop {dup_row_count} duplicate rows but dropped {actual_dropped}. "
    )

## Check for NULL Values

In [0]:
for column in df_weather.columns:

    null_count = df_weather.select(F.sum(F.col(column).isNull().cast("int")).alias("count")).first()["count"]

    if null_count > 0:

        print(f"Found {null_count} NULLs in {column} column. Dropping NULL rows.")
        df_weather = df_weather.filter(F.col(column).isNotNull())
    else:
        print(f"Found {null_count} nulls in {column} column")

## Update time column data type to date

In [0]:
df_weather = df_weather.withColumn("time", F.to_date(F.col("time")))

## Renaming Columns

In [0]:
# Rename columns

rename_columns = {
    "time": "date", 
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "wind_speed_10m_max": "wind_speed_max",
    "wind_gusts_10m_max": "wind_gusts_max"
}

In [0]:
# Updating column names
for old_column, new_column in rename_columns.items():
    df_weather = df_weather.withColumnRenamed(old_column, new_column)

# Write Into Silver Table

In [0]:
df_weather.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_travel.weather")

In [0]:
%sql
SELECT *
FROM silver_travel.weather
LIMIT 100